# 07 - Seed Sensitivity Analysis
Aggregate MLP/LSTM/PatchTST results across multiple seeds and visualize mean ? std.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path('saved_results')
SEEDS = [42, 123, 2024]
METRICS = [
    'overall_mae',
    'warm_mae',
    'cold_mae',
    'overall_cvrmse',
    'overall_wape'
]
DISPLAY_STRATEGIES = [
    '1. FTL', '2. Personalized-FL', '3. Progressive Unfreezing',
    '4. Instance-TL', '5. Fed-SimTL', '6. FedMetaTL'
]

print('Results dir:', RESULTS_DIR.resolve())
print('Seeds:', SEEDS)


In [ ]:
# Load standardized seed summaries
pattern = re.compile(r'^(mlp|lstm|patchtst)_seed_(\d+)_summary\.csv$', re.IGNORECASE)
frames = []
for p in RESULTS_DIR.glob('*_seed_*_summary.csv'):
    m = pattern.match(p.name)
    if not m:
        continue
    model = m.group(1).upper().replace('PATCHTST', 'PatchTST')
    seed = int(m.group(2))
    if seed not in SEEDS:
        continue
    df = pd.read_csv(p)
    if 'model' not in df.columns:
        df['model'] = model
    if 'seed' not in df.columns:
        df['seed'] = seed
    frames.append(df)

if not frames:
    raise FileNotFoundError('No standardized seed summary CSVs found. Run export cells in 03/04/06 first.')

all_seed_results = pd.concat(frames, ignore_index=True)
all_seed_results['model'] = all_seed_results['model'].replace({'MLP':'MLP','LSTM':'LSTM','PATCHTST':'PatchTST'})
all_seed_results = all_seed_results.sort_values(['model','strategy','seed']).reset_index(drop=True)
print('Loaded rows:', len(all_seed_results))
display(all_seed_results.head(20))


In [ ]:
# Aggregate mean/std over seeds
required_cols = {'model', 'seed', 'strategy', *METRICS}
missing = [c for c in required_cols if c not in all_seed_results.columns]
if missing:
    raise ValueError(f'Missing columns for aggregation: {missing}')

for metric in METRICS:
    agg = (all_seed_results
           .groupby(['model', 'strategy'])[metric]
           .agg(['mean', 'std', 'count'])
           .reset_index())

    agg.columns = [
        'model',
        'strategy',
        f'{metric}_mean',
        f'{metric}_std',
        'n_seeds'
    ]

    agg = agg[agg['strategy'].isin(DISPLAY_STRATEGIES)]
    agg = agg.sort_values(['model', 'strategy']).reset_index(drop=True)

    print(f'\nMetric: {metric}')
    display(agg)

    out_path = RESULTS_DIR / f'seed_agg_{metric}.csv'
    agg.to_csv(out_path, index=False)
    print('Saved:', out_path)

In [ ]:
# Publication table: average +/- standard deviation over seeds
METRIC_LABELS = {
    'overall_mae': 'Overall MAE',
    'warm_mae': 'Warm MAE',
    'cold_mae': 'Cold MAE',
    'overall_cvrmse': 'CVRMSE (%)',
    'overall_wape': 'WAPE (%)',
}

def mean_std_text(mean, std, digits=3):
    if pd.isna(mean):
        return ''
    if pd.isna(std):
        return f'{mean:.{digits}f}'
    return f'{mean:.{digits}f} +/- {std:.{digits}f}'

summary_rows = []
for (model, strategy), d in all_seed_results.groupby(['model', 'strategy']):
    row = {
        'Model': model,
        'Strategy': strategy,
        'Seeds': ', '.join(str(int(s)) for s in sorted(d['seed'].dropna().unique())),
        'N seeds': int(d['seed'].nunique()),
    }
    for metric in METRICS:
        row[METRIC_LABELS[metric]] = mean_std_text(d[metric].mean(), d[metric].std(ddof=1), digits=3)
        row[f'{metric}_mean'] = d[metric].mean()
        row[f'{metric}_std'] = d[metric].std(ddof=1)
    summary_rows.append(row)

seed_mean_std_table = pd.DataFrame(summary_rows)
model_order = ['MLP', 'LSTM', 'PatchTST']
strategy_order = [
    'C. Centralized',
    '0. Local-only',
    '1b. FedAvg (No Pretrain)',
    '1. FTL',
    '2. Personalized-FL',
    '3. Progressive Unfreezing',
    '4. Instance-TL',
    '5. Fed-SimTL',
    '6. FedMetaTL',
]
seed_mean_std_table['Model'] = pd.Categorical(seed_mean_std_table['Model'], categories=model_order, ordered=True)
seed_mean_std_table['Strategy'] = pd.Categorical(seed_mean_std_table['Strategy'], categories=strategy_order, ordered=True)
seed_mean_std_table = seed_mean_std_table.sort_values(['Model', 'Strategy']).reset_index(drop=True)
seed_mean_std_table['Model'] = seed_mean_std_table['Model'].astype(str)
seed_mean_std_table['Strategy'] = seed_mean_std_table['Strategy'].astype(str)

display_cols = ['Model', 'Strategy', 'Seeds', 'N seeds'] + list(METRIC_LABELS.values())
display(seed_mean_std_table[display_cols])

csv_path = RESULTS_DIR / 'seed_mean_std_publication_table.csv'
tex_path = RESULTS_DIR / 'seed_mean_std_publication_table.tex'
seed_mean_std_table[display_cols].to_csv(csv_path, index=False)
latex_df = seed_mean_std_table[display_cols].copy()
for col in METRIC_LABELS.values():
    latex_df[col] = latex_df[col].str.replace('+/-', r'$\\pm$', regex=False)

def latex_escape(value):
    text = str(value)
    text = text.replace('\\', r'\\textbackslash{}')
    for old, new in {'_': r'\\_', '%': r'\\%', '&': r'\\&', '#': r'\\#'}.items():
        text = text.replace(old, new)
    return text.replace('$\\\\pm$', r'$\\pm$')

with open(tex_path, 'w', encoding='utf-8') as f:
    f.write('\\begin{table}[htbp]\n\\centering\n')
    f.write('\\caption{Average performance across random seeds reported as mean $\\\\pm$ standard deviation.}\n')
    f.write('\\label{tab:seed_mean_std_results}\n')
    f.write('\\begin{tabular}{llrrccccc}\n\\hline\n')
    f.write(' & '.join(latex_escape(c) for c in display_cols) + r' \\ \hline' + '\n')
    for _, row in latex_df.iterrows():
        f.write(' & '.join(latex_escape(row[c]) for c in display_cols) + r' \\' + '\n')
    f.write('\\hline\n\\end{tabular}\n\\end{table}\n')

print('Saved:', csv_path)
print('Saved:', tex_path)

expected = set(SEEDS)
available = (all_seed_results.groupby(['model', 'strategy'])['seed']
             .apply(lambda s: set(int(x) for x in s.dropna().unique()))
             .reset_index(name='available_seeds'))
missing_seed_rows = []
for _, r in available.iterrows():
    missing = sorted(expected - r['available_seeds'])
    if missing:
        missing_seed_rows.append({
            'model': r['model'],
            'strategy': r['strategy'],
            'missing_seeds': missing,
        })

if missing_seed_rows:
    print('\nWarning: some model/strategy rows do not include all requested seeds:')
    display(pd.DataFrame(missing_seed_rows))


In [ ]:

# Plot 1: per-model bars over strategies (mean Â± std across seeds)

models = ['MLP', 'LSTM', 'PatchTST']

for metric in METRICS:

    # Load the corresponding aggregated dataframe
    agg = pd.read_csv(RESULTS_DIR / f'seed_agg_{metric}.csv')

    for model in models:

        d = agg[agg['model'] == model].copy()

        if d.empty:
            continue

        d['strategy'] = pd.Categorical(
            d['strategy'],
            categories=DISPLAY_STRATEGIES,
            ordered=True
        )

        d = d.sort_values('strategy')

        plt.figure(figsize=(11, 4.8))

        x = np.arange(len(d))
        y = d[f'{metric}_mean'].values
        e = np.nan_to_num(d[f'{metric}_std'].values, nan=0.0)

        plt.bar(
            x,
            y,
            yerr=e,
            capsize=6,
            color='#ef746c',
            edgecolor='#ef746c',
            alpha=0.95
        )

        # value labels
        for xi, yi in zip(x, y):
            plt.text(
                xi,
                yi,
                f'{yi:.3f}',
                ha='center',
                va='bottom',
                fontsize=10,
                color='#e85f57',
                fontweight='bold'
            )

        plt.xticks(x, d['strategy'], rotation=22, ha='right')

        plt.ylabel(metric.replace('_', ' ').title())

        plt.title(
            f'{model}: {metric} across strategies '
            f'(mean Â± std over seeds)'
        )

        plt.grid(axis='y', alpha=0.25)

        plt.tight_layout()
        plt.show()


In [ ]:
# Seed sensitivity plots for all strategies and all metrics

for strategy in DISPLAY_STRATEGIES:

    for metric in METRICS:

        d = all_seed_results[
            all_seed_results['strategy'] == strategy
        ].copy()

        if d.empty:
            print(f'No rows for strategy: {strategy}')
            continue

        d = d[d['seed'].isin(SEEDS)].sort_values(['seed', 'model'])

        pivot = d.pivot_table(
            index='seed',
            columns='model',
            values=metric,
            aggfunc='mean'
        )

        preferred_order = ['MLP', 'LSTM', 'PatchTST']

        existing_cols = [c for c in preferred_order if c in pivot.columns]
        pivot = pivot[existing_cols]

        pivot.plot(
            kind='bar',
            figsize=(10.5, 4.8),
            color=['#ef746c', '#69a2ff', '#6bbf8f'][:len(existing_cols)]
        )

        plt.title(
            f'Seed sensitivity for {strategy} ({metric})'
        )

        plt.xlabel('Seed')

        plt.ylabel(
            metric.replace('_', ' ').title()
        )

        plt.grid(axis='y', alpha=0.25)

        plt.tight_layout()
        plt.show()

        display(pivot.reset_index())

## How To Run
1. In each model notebook, set `SEED` (e.g. `SEED=42`), run training/evaluation, then run the final standardized export cell.
2. Repeat for all seeds in `SEEDS`.
3. Run this notebook to aggregate and plot.
